In [ ]:
!pip install -q crewai
!pip install -q openai
!pip install -q unstructured
!pip install -q tools
!pip install -q tenacity==8.3.0
!pip install -q langchain
!pip install -q langchain_groq
!pip install -q cohere
!pip install -q langchain_community
!pip install -q 'crewai[tools]'

In [ ]:
from crewai import Agent, Task, Crew
from langchain_community.chat_models import ChatCohere
from langchain_openai import OpenAI
from langchain_groq import ChatGroq


In [ ]:
#warning control
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os

COHERE_API_KEY="ieLPQ8jIDR7czKpyAcRYPJTdjjP27AAVwR7Gvwzs"
OPENAI_API_KEY="sk-crew-ai-gqQfhuNQaIn8AKoIRg8UT3BlbkFJfEam04Z3SQEPwjWabC4Y"
GROQ_API_KEY="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"
SERPER_API_KEY= "25b084c4ee79d46c1866395746dbdf145c1729d9"


os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
os.environ['COHERE_API_KEY'] = COHERE_API_KEY
os.environ['GROQ_API_KEY'] = GROQ_API_KEY
os.environ['SERPER_API_KEY'] = SERPER_API_KEY

#Tools
from crewai_tools import (
    SerperDevTool,
    WebsiteSearchTool
)

search_tool = SerperDevTool()
web_search_tool = WebsiteSearchTool()

#LLMs

cohere = ChatCohere(cohere_api_key=COHERE_API_KEY,
                    temperaature= 0.3)
openai = OpenAI(api_key=OPENAI_API_KEY)
groq = ChatGroq(
                temperature=0,
                groq_api_key=GROQ_API_KEY,
                model_name="mixtral-8x7b-32768"
            )


In [ ]:
#Agent
business_portfolio_analyst = Agent(
                         role='Business Portfolio Analyst',
                         goal="""Identify and list the companies and Job Titles within {report} that are responsible for or will benefit from {sender} or {briefDes},
                                       and the supervisor in charge of the Job Titles.""",
                         backstory="""You are a Business Portfolio Analyst that identifies relevant companies and job titles within {report}
                                      that would benefit from {sender}, or {briefDes}. You analyze information from the Business Analyst and based on its output, make this decision.
                                      Your mission is to enhance cold email campaigns by targeting Job Titles and the key decision makers of these Job titles who are the final decision-makers.""",
                        # tools=[
                        #         search_tool
                        #       ],
                        allow_delegation=False,
                        verbose=True,
                        llm=cohere,
                       )

In [ ]:
profile = Task (
       description= """
                       Identify and list the Job Titles within {report} that are responsible for or will benefit from {sender}, {briefDes}.
                       Using the information provided by the Business Portfolio Analyst, identify the Job Titles' supervisors of the subniches.
                       Gather information about the relevant job titles, their responsibilities, and how they will benefit from the offer. Identify their supervisors and understand the subniches they oversee.
                       This analysis should provide a detailed overview of the organizational structure, including the key job titles, their roles, and their supervisors.
                       Focus on responsibilities, potential benefits from the offer, and the overall hierarchy within the company.
                       The final answer must be a comprehensive report on the identified job titles and their supervisors, rich in organizational insights and practical information,
                       tailored to understand the company's internal structure and how it relates to the offer.
                   """,
            expected_output="A comprehensive detail of the Job Titles' supervisors of the subniches that will benefit from {sender}, or {briefDes}",
            agent=business_portfolio_analyst)

In [ ]:
#Crew
crew = Crew(
    agents=[business_portfolio_analyst],
    tasks=[profile],
    verbose=True,
)

In [ ]:
#Execute Crew
result = crew.kickoff(inputs={
    "sender": "Scaletific",
    "report": """
                 Companies and Subniches:

                 1. **Tesla, Inc. (Subniche: Electric Vehicles and Sustainable Transport):**
                    - Tesla is a renowned automobile manufacturer specializing in electric vehicles (EVs) and clean energy solutions. Its subniche focuses on sustainable transport, advanced battery technology, and autonomous driving.
                    - **Pain Points:**
                    - Battery Range and Charging Infrastructure: Improving battery efficiency and expanding charging networks to address range anxiety.
                    - Production Capacity: Meeting the high demand for EVs and scaling production while maintaining quality.
                    - Autonomous Driving Regulations: Navigating legal and safety standards for self-driving features.

                 2. **Toyota Motor Corporation (Subniche: Hybrid Vehicles and Lean Manufacturing):**
                     - Toyota is known for its hybrid vehicle technology and efficient production processes. Its subniche involves hybrid engine systems, fuel efficiency, and lean manufacturing practices.
                     - **Pain Points:**
                     - Hybrid Technology Advancements: Continuously enhancing hybrid systems to compete with fully electric vehicles.
                     - Supply Chain Management: Optimizing supply chains for sustainable materials and components.
                     - Global Market Adaptation: Tailoring vehicles to diverse regional preferences and regulations.

                 3. **Mercedes-Benz (Subniche: Luxury Automobiles and Advanced Driver Assistance):**
                    - Mercedes-Benz operates in the luxury car segment, emphasizing comfort, performance, and cutting-edge driver assistance systems.
                    - **Pain Points:**
                    - Personalized Customer Experience: Customizing features and services to cater to high-end clientele.
                    - Autonomous Driving and Safety: Developing advanced driver assistance systems while adhering to legal frameworks.
                    - Digital Transformation: Integrating digital technologies for enhanced customer engagement.

                 4. **Harley-Davidson (Subniche: Motorcycles and Lifestyle Brand):**
                    - Harley-Davidson is a motorcycle manufacturer with a strong brand identity and a loyal customer base. Its subniche includes motorcycle design, customization, and a lifestyle-oriented community.
                    - **Pain Points:**
                    - Electric Motorcycle Transition: Adapting to the rise of electric motorcycles while maintaining brand heritage.
                    - Customer Engagement: Creating digital experiences and communities to attract younger audiences.
                    - Global Market Presence: Expanding internationally while preserving brand authenticity.

                5. **Uber Technologies, Inc. (Subniche: Ride-Hailing and Mobility Services):**
                   - Uber operates in the mobility services subniche, offering ride-hailing, food delivery, and logistics solutions.
                   - **Pain Points:**
                   - Regulatory Compliance: Navigating legal challenges and local regulations in various markets.
                   - Driver and Rider Safety: Implementing safety measures and background checks.
                   - Efficient Logistics: Optimizing routes and fleet management for cost-effective operations.
                   - Customer Experience: Enhancing the app experience and loyalty programs.

                   ## Scaletific Services Beneficial to the Automobile Industry:

                   1. **Hyperautomation:**
                        - Process Automation: Scaletific can assist in automating various processes, from manufacturing to customer service, reducing manual errors and increasing efficiency.
                        - Quality Control: Automated systems can ensure consistent quality, especially in production and supply chain management.

                   2. **Data-Driven Insights:**
                        - Market Analysis: Providing data analytics for market segmentation, customer preferences, and industry trends to inform strategic decisions.
                        - Predictive Maintenance: Using data to predict vehicle maintenance needs, reducing downtime and costs.

                   3. **AI and Machine Learning:**
                        - Autonomous Driving Development: Support in AI algorithms for self-driving vehicles, a key focus for many automobile companies.
                         - Personalized Recommendations: Tailoring customer experiences through AI-driven suggestions.

                   4. **Digital Transformation:**
                        - Digital Platform Development: Creating digital platforms for customer engagement, sales, and service bookings.
                        - Online Community Building: Helping companies like Harley-Davidson foster digital communities.

                    This report highlights the diverse nature of the automobile industry and the unique challenges and opportunities within each subniche. Scaletific's services can significantly contribute to these companies' operations, customer satisfaction, and technological advancements.

                    """,
                   "briefDes":"We offer hyperautomation"
     })

 [2024-08-30 13:38:48][DEBUG]: == Working Agent: Business Portfolio Analyst
 [2024-08-30 13:38:48][INFO]: == Starting Task:  
                       Identify and list the Job Titles within  
                 Companies and Subniches:

                 1. **Tesla, Inc. (Subniche: Electric Vehicles and Sustainable Transport):**
                    - Tesla is a renowned automobile manufacturer specializing in electric vehicles (EVs) and clean energy solutions. Its subniche focuses on sustainable transport, advanced battery technology, and autonomous driving.
                    - **Pain Points:**
                    - Battery Range and Charging Infrastructure: Improving battery efficiency and expanding charging networks to address range anxiety.
                    - Production Capacity: Meeting the high demand for EVs and scaling production while maintaining quality.
                    - Autonomous Driving Regulations: Navigating legal and safety standards for self-driving features.

   

NameError: name 'result' is not defined